# Lab 54 (solution): Production durable backends

Reference implementation. [Lab 50](../../50-closing-the-failure-loop/) built the dead-letter queue as a file with a lock and noted the real thing is Redis Streams or SQS, *same lease/ack/retention contract*. This makes that literal: one contract, two production backends, and the Lab 50 redelivery worker driving either unchanged.

The real client code is shown behind guarded imports; behavior is verified against in-process fakes that mirror each system's semantics, so the same contract test passes on both.

## Step 0: Setup

In [ ]:
from backends import (RedisStreamQueue, SQSQueue, run_contract,
                      FakeRedisStreams, FakeSQS)
# Lab 50's DurableQueue was a file with a lock; production is Redis Streams or SQS, "same
# lease/ack/retention contract." Here that is literal: two backends, one contract.
print("backends:", RedisStreamQueue.__name__, "(consumer-group PEL) and", SQSQueue.__name__, "(visibility timeout)")

## Step 1: One contract, two sets of primitives

In [ ]:
# The contract maps onto each system's native primitives:
print("contract op   | Redis Streams           | SQS")
print("-"*70)
for op, r, s in [("enqueue","XADD","SendMessage"),
                 ("lease","XREADGROUP (-> PEL)","ReceiveMessage (VisibilityTimeout)"),
                 ("ack","XACK + XDEL","DeleteMessage"),
                 ("reclaim expired","XAUTOCLAIM (min-idle)","timeout lapses -> visible again"),
                 ("give up","PEL deliveries > max -> DLQ","ReceiveCount > max -> redrive to DLQ")]:
    print(f"{op:13s} | {r:23s} | {s}")

## Step 2: The contract on Redis Streams

In [ ]:
# Drive the Lab 50 redelivery contract against Redis Streams (over a fake mirroring XADD /
# XREADGROUP / XACK / XAUTOCLAIM). Scenario: m0 ok, m1 transient (recovers on redelivery),
# m2 permanent (gives up to the dead set).
def make_send():
    seen=set()
    def send(p):
        m=p["metric"]
        if m == "m2":
            raise RuntimeError("permanently down")
        if m == "m1" and m not in seen:
            seen.add(m)
            raise RuntimeError("transient blip")
    return send
print("redis-streams:", run_contract(RedisStreamQueue(max_deliveries=3), make_send()))

## Step 3: The same contract on SQS

In [ ]:
# The same contract, the same worker, against SQS (over a fake mirroring SendMessage /
# ReceiveMessage with a visibility timeout / DeleteMessage / redrive).
print("sqs:          ", run_contract(SQSQueue(max_receives=3), make_send()))
print("\nIdentical observable outcome from two very different primitives - that is the point of")
print("coding to the contract, not the backend.")

## Step 4: Swapping in a real client

In [ ]:
# In production you swap the fake for a real client and nothing else changes:
print("redis:  RedisStreamQueue(redis.Redis.from_url(os.environ['REDIS_URL']))")
print("sqs:    SQSQueue(boto3.client('sqs'))   # with a redrive policy + DLQ configured")
print("\nThe redelivery worker from Lab 50 calls enqueue/lease/ack/reclaim_expired and never")
print("learns which backend it is talking to.")

## What you built

The Lab 50 durable-queue contract on two production backends. `backends.py` defines one interface - `enqueue` / `lease` / `ack` / `reclaim_expired` / `pending` / `dead` - and implements it twice: `RedisStreamQueue` over a consumer group (XADD enqueues, XREADGROUP leases into the Pending Entries List, XACK acks, XAUTOCLAIM reclaims entries idle past the lease, the PEL delivery count drives give-up) and `SQSQueue` over a visibility timeout (SendMessage / ReceiveMessage with VisibilityTimeout / DeleteMessage, and a redrive policy to a DLQ past maxReceiveCount). The contract test produces the *identical* end state on both - m0 acked, m1 recovered on redelivery, m2 given up to the dead set - from two systems whose primitives have nothing in common.

**Where this simplifies:** the backends run against in-process fakes (`FakeRedisStreams`, `FakeSQS`) that mirror the exact operations used, so the lab is deterministic and offline; production passes a real `redis.Redis(...)` or `boto3.client('sqs')` and the worker is unchanged. The fakes implement only the operations the contract needs - a real deployment also configures stream `MAXLEN` / consumer-group creation, SQS redrive policy and DLQ ARNs, and the at-least-once duplicate handling both systems imply (acks can race a reclaim, so the consumer must be idempotent). Retention here is XDEL-on-ack / DeleteMessage-on-ack; real systems add a time-based retention floor.